# Taller de herramientas: scikit-learn y NVIDIA NIM

**SI3003 · Sesiones 6–9 · 60–90 minutos incluyendo las preguntas conceptuales.**

Ejecute las celdas en orden. Solo cambiará la profundidad de un árbol y el texto de tres consultas. El código está preparado; no debe implementar algoritmos. Puede usar Google Colab con CPU.

In [ ]:
%pip -q install scikit-learn openai

## 1. Clasificación con scikit-learn

Iris contiene 150 flores de tres especies, representadas por cuatro medidas. Reservamos datos para entrenar, elegir entre dos profundidades con validación y evaluar finalmente en prueba. Ejecute esta celda una sola vez; mantenga las particiones durante la comparación.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X_trainval, X_test, y_trainval, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print("Características:", list(iris.feature_names))
print("Clases:", list(iris.target_names))
print("Tamaños train / validación / prueba:", len(y_train), len(y_val), len(y_test))
resultados = {}
modelos = {}

### Dos experimentos

Ejecute primero con `profundidad = 2`. Luego cambie únicamente ese valor a `5` y vuelva a ejecutar la siguiente celda. Ambos resultados quedarán guardados.

In [ ]:
profundidad = 2  # CAMBIE A 5 en la segunda ejecución

modelo = DecisionTreeClassifier(max_depth=profundidad, random_state=42)
modelo.fit(X_train, y_train)
train_acc = accuracy_score(y_train, modelo.predict(X_train))
val_acc = accuracy_score(y_val, modelo.predict(X_val))
modelos[profundidad] = modelo
resultados[profundidad] = (train_acc, val_acc)
print(f"Profundidad: {profundidad}")
print(f"Exactitud de entrenamiento: {train_acc:.3f}")
print(f"Exactitud de validación: {val_acc:.3f}")

**Complete la tabla con los resultados que obtuvo:**

| Profundidad | Exactitud entrenamiento | Exactitud validación |
|---|---|---|
| 2 | | |
| 5 | | |

Elija la profundidad con mayor exactitud de validación. En empate, elija 2. Escriba ese valor en la siguiente celda y evalúe una sola vez en prueba.

In [ ]:
profundidad_elegida = 2  # ESCRIBA 2 o 5 según validación

if not {2, 5}.issubset(resultados):
    raise ValueError("Ejecute primero el experimento con profundidad 2 y luego con 5.")
mejor_segun_validacion = max([2, 5], key=lambda d: (resultados[d][1], -d))
if profundidad_elegida != mejor_segun_validacion:
    raise ValueError("Revise la exactitud de validación; en empate prefiera profundidad 2.")
modelo_final = modelos[profundidad_elegida]
predicciones = modelo_final.predict(X_test)
print(f"Exactitud de prueba: {accuracy_score(y_test, predicciones):.3f}")
for real, predicha in zip(y_test[:5], predicciones[:5]):
    print("Real:", iris.target_names[real], "| Predicción:", iris.target_names[predicha])

**Análisis (máximo 5 líneas):** ¿qué modelo eligió?, ¿aumentar la profundidad ayudó?, ¿por qué un resultado perfecto en entrenamiento no garantiza generalización?

_Escriba aquí._

## 2. Consultas a un LLM con NVIDIA NIM

1. Abra [NVIDIA Build](https://build.nvidia.com/) e inicie sesión.
2. Busque el modelo indicado por el docente y abra su ejemplo de API. Obtenga una API key mediante la opción disponible en la plataforma.
3. El modelo de ejemplo es `meta/llama-3.1-8b-instruct`. Si el docente indica otro modelo habilitado en su cuenta, sustituya `MODELO` por su identificador exacto.
4. Ejecute la celda e introduzca la clave cuando aparezca el campo oculto. No la pegue en el código. Se utiliza el servicio alojado por NVIDIA, sin instalar un servidor NIM ni descargar pesos.

Referencia: [API de NVIDIA para Llama 3.1 8B](https://docs.api.nvidia.com/nim/reference/meta-llama-3_1-8b-infer).

In [ ]:
from getpass import getpass
from openai import OpenAI

MODELO = "meta/llama-3.1-8b-instruct"
api_key = getpass("API key de NVIDIA (entrada oculta): ").strip()
if not api_key:
    raise ValueError("Ingrese una API key de NVIDIA.")
cliente = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
    timeout=60.0,
    max_retries=1,
)

# Función preparada: solo necesita llamarla con el texto de su consulta.
def consultar(texto):
    respuesta = cliente.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Responde en español de forma clara y breve."},
            {"role": "user", "content": texto},
        ],
        temperature=0.2,
        max_tokens=350,
        stream=False,
    )
    return respuesta.choices[0].message.content

print("Cliente configurado. La primera consulta comprobará el acceso al modelo.")

### Consulta 1: pregunta general

Ejecute la celda y lea la respuesta. Cada llamada envía una consulta independiente, sin historial de las anteriores.

In [ ]:
consulta_1 = "¿Qué es el sobreajuste?"
respuesta_1 = consultar(consulta_1)
print(respuesta_1)

### Consulta 2: mejorar las instrucciones

Reemplace el texto vacío por su consulta. Pida una explicación para principiantes, un ejemplo con árboles de decisión y un máximo de 100 palabras. Mantenga el resto del código.

In [ ]:
consulta_2 = ""  # ESCRIBA AQUÍ SU CONSULTA MEJORADA
if not consulta_2.strip():
    raise ValueError("Escriba su consulta antes de ejecutar.")
respuesta_2 = consultar(consulta_2)
print(respuesta_2)

### Consulta 3: verificar una afirmación

Ejecute y compruebe la explicación con las diapositivas y su experimento de scikit-learn. Si incluye afirmaciones equivocadas, señálelas; no asuma que la respuesta es correcta.

In [ ]:
consulta_3 = (
    "Un árbol con 100 % de exactitud en entrenamiento siempre será el mejor "
    "en datos nuevos. ¿Es correcta esta afirmación? Justifica tu respuesta."
)
respuesta_3 = consultar(consulta_3)
print(respuesta_3)

**Análisis (máximo 5 líneas):** compare las respuestas 1 y 2: claridad, ejemplo y cumplimiento del límite de palabras. ¿La respuesta 3 concuerda con lo estudiado? ¿Por qué debe verificarla? ¿Estas consultas entrenan el modelo o ejecutan inferencia?

_Escriba aquí._

### Si falla la conexión

- **401:** revise la API key y vuelva a ejecutar la configuración.
- **403/404:** revise el identificador del modelo y su acceso en NVIDIA Build.
- **429:** se alcanzó un límite de solicitudes o de uso; revise su cuenta y vuelva a intentar más tarde.
- **Timeout/5xx:** vuelva a intentar la consulta más tarde.

No incluya su clave al compartir un error. Guarde el notebook con las respuestas visibles; `getpass` no guarda el valor introducido en las celdas.